# AgriSubsidy Random Forest Training with SMOTE

This notebook trains a Random Forest classifier using `Outcome Cause` and applies SMOTE **only to the training data**. The external validation dataset is kept untouched.

# Imports

In [ ]:
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
)
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE

# Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

TRAIN_FILE = "datasets/subsidy_dataset2.xlsx"
VALIDATION_FILE = "datasets/subsidy_validation_datasets2.xlsx"

TARGET = "Effectiveness Label"

RANDOM_STATE = 42

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Load datasets

In [ ]:
# ============================================================
# LOAD DATASETS
# ============================================================

print("=" * 70)
print("LOADING DATASETS")
print("=" * 70)

train_df = pd.read_excel(TRAIN_FILE)
validation_df = pd.read_excel(VALIDATION_FILE)

print(f"Training dataset shape:   {train_df.shape}")
print(f"Validation dataset shape: {validation_df.shape}")

# Clean columns and validate required columns

In [ ]:
# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

train_df.columns = train_df.columns.str.strip()
validation_df.columns = validation_df.columns.str.strip()


# ============================================================
# REQUIRED COLUMNS
# ============================================================

categorical_cols = [
    "Subsidy Type",
    "Pest",
    "Calamity",
    "Outcome Cause",
]

numerical_cols = [
    "Farm Size (ha)",
    "Crop Yield Before",
    "Crop Yield After",
    "Income Before",
    "Income After",
    "Feedback Score",
]

required_columns = categorical_cols + numerical_cols + [TARGET]

missing_train = [
    col for col in required_columns
    if col not in train_df.columns
]

missing_validation = [
    col for col in required_columns
    if col not in validation_df.columns
]

if missing_train:
    raise ValueError(
        f"Missing training columns: {missing_train}"
    )

if missing_validation:
    raise ValueError(
        f"Missing validation columns: {missing_validation}"
    )

# Clean missing values

In [ ]:
# ============================================================
# CLEAN MISSING VALUES
# ============================================================

for col in categorical_cols:
    train_df[col] = train_df[col].fillna("None").astype(str)
    validation_df[col] = validation_df[col].fillna("None").astype(str)

for col in numerical_cols:
    train_df[col] = pd.to_numeric(
        train_df[col], errors="coerce"
    )
    validation_df[col] = pd.to_numeric(
        validation_df[col], errors="coerce"
    )

    median_value = train_df[col].median()

    train_df[col] = train_df[col].fillna(median_value)
    validation_df[col] = validation_df[col].fillna(median_value)


# Remove rows where the target is missing
train_df = train_df.dropna(subset=[TARGET]).copy()
validation_df = validation_df.dropna(subset=[TARGET]).copy()

# Inspect class distribution

In [ ]:
# ============================================================
# DISPLAY ORIGINAL CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("ORIGINAL TRAINING CLASS DISTRIBUTION")
print("=" * 70)

print(train_df[TARGET].value_counts())
print(
    train_df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .astype(str)
    .add("%")
)

# Features and target

In [ ]:
# ============================================================
# FEATURES AND TARGET
# ============================================================

X = train_df[categorical_cols + numerical_cols]
y = train_df[TARGET]

X_validation = validation_df[categorical_cols + numerical_cols]
y_validation = validation_df[TARGET]

# Train/test split

In [ ]:
# ============================================================
# TRAIN / INTERNAL TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("\n" + "=" * 70)
print("TRAIN / INTERNAL TEST SPLIT")
print("=" * 70)

print(f"Training rows:     {len(X_train)}")
print(f"Internal test rows:{len(X_test)}")

# Preprocessing

In [ ]:
# ============================================================
# PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_cols,
        ),
        (
            "numerical",
            "passthrough",
            numerical_cols,
        ),
    ]
)


# Fit preprocessing ONLY on training data.
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test and external validation using the same fitted
# preprocessing object.
X_test_processed = preprocessor.transform(X_test)
X_validation_processed = preprocessor.transform(X_validation)

# SMOTE

In [ ]:
# ============================================================
# SMOTE
# ============================================================

print("\n" + "=" * 70)
print("SMOTE")
print("=" * 70)

print("Before SMOTE:")
print(pd.Series(y_train).value_counts().sort_index())

# SMOTE is applied ONLY to the training portion.
#
# The validation and internal test sets are NOT oversampled.
#
# This prevents synthetic validation samples from influencing
# the evaluation of the model.

class_counts = pd.Series(y_train).value_counts()

# SMOTE's default k_neighbors=5 requires at least 6 samples
# in every minority class.
min_class_count = class_counts.min()

if min_class_count >= 6:
    smote_neighbors = min(5, min_class_count - 1)

    smote = SMOTE(
        random_state=RANDOM_STATE,
        k_neighbors=smote_neighbors,
    )

    X_train_smote, y_train_smote = smote.fit_resample(
        X_train_processed,
        y_train,
    )

    print("\nAfter SMOTE:")
    print(pd.Series(y_train_smote).value_counts().sort_index())

else:
    print(
        "\nSMOTE skipped because the smallest class has "
        f"only {min_class_count} sample(s)."
    )

    X_train_smote = X_train_processed
    y_train_smote = y_train

# Random Forest and hyperparameter search

In [ ]:
# ============================================================
# RANDOM FOREST
# ============================================================

print("\n" + "=" * 70)
print("RANDOM FOREST")
print("=" * 70)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight=None,
)


# ============================================================
# RANDOMIZED SEARCH
# ============================================================

param_distributions = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [None, 10, 15, 20, 25, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
}

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="accuracy",
    cv=5,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("Starting RandomizedSearchCV...")

search.fit(
    X_train_smote,
    y_train_smote,
)

best_model = search.best_estimator_

print("\nBest Parameters:")
print(search.best_params_)

print(f"\nBest CV Accuracy: {search.best_score_:.4f}")

# Internal test evaluation

In [ ]:
# ============================================================
# INTERNAL TEST EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("INTERNAL TEST EVALUATION")
print("=" * 70)

y_test_pred = best_model.predict(X_test_processed)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred,
)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        zero_division=0,
    )
)

test_cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=best_model.classes_,
)

print("\nConfusion Matrix:")
print(test_cm)

# External validation

In [ ]:
# ============================================================
# EXTERNAL VALIDATION DATASET
# ============================================================

print("\n" + "=" * 70)
print("EXTERNAL VALIDATION DATASET")
print("=" * 70)

y_validation_pred = best_model.predict(
    X_validation_processed
)

validation_accuracy = accuracy_score(
    y_validation,
    y_validation_pred,
)

print(
    f"Validation Accuracy: "
    f"{validation_accuracy:.4f}"
)

print(
    f"Validation Accuracy: "
    f"{validation_accuracy * 100:.2f}%"
)

print("\nValidation Classification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred,
        zero_division=0,
    )
)

validation_cm = confusion_matrix(
    y_validation,
    y_validation_pred,
    labels=best_model.classes_,
)

print("\nValidation Confusion Matrix:")
print(validation_cm)

# Confusion matrix

In [ ]:
# ============================================================
# CONFUSION MATRIX IMAGE
# ============================================================

plt.figure(figsize=(8, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=validation_cm,
    display_labels=best_model.classes_,
)

disp.plot(
    values_format="d",
    ax=plt.gca(),
)

plt.title("Random Forest Validation Confusion Matrix")
plt.tight_layout()

plt.savefig(
    "validation_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(
    "\nSaved: validation_confusion_matrix.png"
)

# Feature importance

In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)

feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best_model.feature_importances_,
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False,
)

print("\nTop 20 Features:")
print(
    feature_importance.head(20).to_string(
        index=False
    )
)

feature_importance.to_excel(
    "feature_importance.xlsx",
    index=False,
)

print("\nSaved: feature_importance.xlsx")

# Save model

In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================

model_package = {
    "model": best_model,
    "preprocessor": preprocessor,
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "target": TARGET,
    "classes": best_model.classes_,
}

joblib.dump(
    model_package,
    "subsidy_random_forest_smote_model.pkl",
)

print(
    "\nSaved: subsidy_random_forest_smote_model.pkl"
)

# Final summary

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(
    f"Training dataset:       {len(train_df):,} rows"
)

print(
    f"Internal test dataset:  {len(X_test):,} rows"
)

print(
    f"External validation:    {len(validation_df):,} rows"
)

print(
    f"SMOTE training rows:    {len(y_train_smote):,} rows"
)

print(
    f"Internal test accuracy:  {test_accuracy * 100:.2f}%"
)

print(
    f"External validation:    {validation_accuracy * 100:.2f}%"
)

print("\nTraining completed successfully.")